# Tag Recommendation Experiment (RecSys 2018)

Notebook simple et modulaire pour reproduire une expérience inspirée de l'article **Semantic-based Tag Recommendation in Scientific Bookmarking Systems**.

- Import des données : **1 cellule**
- Une fonction par cellule
- Visualisations déplacées dans `src/visualization.py`
- Dernière cellule : compilation des résultats + visualisations comparatives

In [1]:
# Cellule 1 - Imports (meme stack que l'article)
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data import load_simple_dataset, keep_top_k_tags, preprocess_text_nltk, text_length_stats
from src.experiment import prepare_train_test, run_all_models
from src.visualization import (
    plot_tag_distribution,
    plot_text_length_distributions,
    plot_model_metrics,
    plot_article_vs_current,
)

pd.set_option("display.max_colwidth", 120)

2026-04-15 18:39:27.762670: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-15 18:39:27.775775: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-15 18:39:27.790287: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-15 18:39:27.794765: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-15 18:39:27.806844: I tensorflow/core/platform/cpu_feature_guar

In [2]:
# Cellule 2 - Paramètres globaux
DATA_PATH = str(PROJECT_ROOT / "data" / "citeulike_top10.csv")
GLOVE_PATH = str(PROJECT_ROOT / "data" / "glove.6B.300d.txt")  # optionnel
TOP_K_TAGS = 100
TEST_SIZE = 0.2
RANDOM_STATE = 42

In [3]:
# Cellule 3 - Import des données (simple, en une cellule)
def import_data(csv_path: str, top_k_tags: int = 10):
    df = load_simple_dataset(csv_path)
    df = keep_top_k_tags(df, top_k=top_k_tags)
    df = preprocess_text_nltk(df)
    return text_length_stats(df)


df = import_data(DATA_PATH, TOP_K_TAGS)
df.head()

,title,abstract,tags,text,tag_list,processed_text,title_words,abstract_words,text_words,num_tags
0,Deep learning for text classification,We study neural approaches for classifying long scientific documents and evaluating quality.,machine learning|classification|deep learning,Deep learning for text classification. We study neural approaches for classifying long scientific documents and eval...,"[machine learning, classification, deep learning]",deep learning text classification study neural approach classifying long scientific document evaluating quality,5,12,13,3
1,Topic models for tags,Latent topic structures improve retrieval and can be used to suggest user tags.,topic modeling|information retrieval,Topic models for tags. Latent topic structures improve retrieval and can be used to suggest user tags.,"[topic modeling, information retrieval]",topic model tag latent topic structure improve retrieval used suggest user tag,4,13,12,2
2,A survey on recommendation systems,This article reviews recommendation methods and discusses ranking and evaluation.,recommendation|survey|ranking,A survey on recommendation systems. This article reviews recommendation methods and discusses ranking and evaluation.,"[recommendation, survey, ranking]",survey recommendation system article review recommendation method discusses ranking evaluation,5,10,10,3
3,Learning representations of documents,Distributed vectors capture semantic meaning from text corpora.,embeddings|representation learning|deep learning,Learning representations of documents. Distributed vectors capture semantic meaning from text corpora.,"[embeddings, representation learning, deep learning]",learning representation document distributed vector capture semantic meaning text corpus,4,8,10,3
4,Attention mechanisms in NLP,Attention allows models to focus on important words in sequence modeling tasks.,attention|nlp|deep learning,Attention mechanisms in NLP. Attention allows models to focus on important words in sequence modeling tasks.,"[attention, nlp, deep learning]",attention mechanism nlp attention allows model focus important word sequence modeling task,4,12,12,3


In [4]:
# Cellule 4 - Fonction de résumé exploration
def summarize_dataset(df: pd.DataFrame) -> pd.DataFrame:
    summary = {
        "num_documents": [len(df)],
        "num_unique_tags": [len({t for tags in df["tag_list"] for t in tags})],
        "avg_title_words": [df["title_words"].mean()],
        "avg_abstract_words": [df["abstract_words"].mean()],
        "avg_tags_per_doc": [df["num_tags"].mean()],
    }
    return pd.DataFrame(summary)


summarize_dataset(df)

,num_documents,num_unique_tags,avg_title_words,avg_abstract_words,avg_tags_per_doc
0,12,24,4.5,10.25,2.916667


In [5]:
# Cellule 5 - Fonction visualisation EDA
def visualize_eda(df: pd.DataFrame):
    fig1 = plot_tag_distribution(df, top_n=TOP_K_TAGS)
    fig2 = plot_text_length_distributions(df)
    plt.show()
    return fig1, fig2


_ = visualize_eda(df)

/workspace/src/visualization.py:16: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=counts.values, y=counts.index, ax=ax, palette="viridis")


In [6]:
# Cellule 6 - Préparation train/test
def make_splits(df: pd.DataFrame):
    return prepare_train_test(
        df,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )


X_train, X_test, y_train, y_test, mlb = make_splits(df)
print("Train:", len(X_train), "Test:", len(X_test), "Labels:", len(mlb.classes_))

Train: 9 Test: 3 Labels: 24


In [8]:
# Cellule 7 - Entraînement de tous les modèles
def train_models(X_train, X_test, y_train, y_test, n_topics: int, glove_path: str):
    return run_all_models(
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        n_topics=n_topics,
        glove_path=glove_path,
    )


metrics_df, predictions = train_models(
    X_train,
    X_test,
    y_train,
    y_test,
    n_topics=TOP_K_TAGS,
    glove_path=GLOVE_PATH,
)
metrics_df

/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:87: UserWarning: Label not 4 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:87: UserWarning: Label not 7 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:87: UserWarning: Label not 9 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:87: UserWarning: Label not 18 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:87: UserWarning: Label not 4 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:87: UserWarning: Label not 7 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:87: UserWarning: Label not 9 is present in all training examples.
 

InternalError: Graph execution error:

Detected at node CudnnRNN defined at (most recent call last):
<stack traces unavailable>
Failed to call DoRnnForward with model config: [rnn_mode, rnn_input_mode, rnn_direction_mode]: 3, 0, 0 , [num_layers, input_size, num_units, dir_count, max_seq_length, batch_size, cell_num_units]: [1, 300, 25, 1, 300, 8, 0] 
	 [[{{node CudnnRNN}}]]
	 [[model_1/bidirectional_1/forward_gru_1/PartitionedCall]] [Op:__inference_train_function_11443]

In [ ]:
# Cellule 8 - Sauvegarde facultative des métriques
def save_metrics(metrics_df: pd.DataFrame, output_csv: str = "data/metrics_results.csv"):
    metrics_df.to_csv(output_csv, index=False)
    return output_csv


save_metrics(metrics_df)

In [ ]:
# Cellule 9 (dernière) - Compilation + visualisations comparatives
def compile_and_compare(metrics_df: pd.DataFrame):
    display(metrics_df.sort_values("micro_f1", ascending=False).reset_index(drop=True))

    fig_models = plot_model_metrics(metrics_df)
    fig_article = plot_article_vs_current(metrics_df)
    plt.show()

    return {
        "best_model": metrics_df.sort_values("micro_f1", ascending=False).iloc[0]["name"],
        "best_micro_f1": float(metrics_df["micro_f1"].max()),
    }


summary = compile_and_compare(metrics_df)
summary